# Sign Language CNN — Training on Google Colab

### Steps before running:
1. Go to **Runtime → Change runtime type → T4 GPU** (free)
2. Zip your `AtoZ_3.1` folder on your PC:
   - Right-click `AtoZ_3.1` → Send to → Compressed (zip)
3. Upload `AtoZ_3.1.zip` using the cell below
4. Run all cells in order (**Runtime → Run all**)
5. Download `cnn8grps_rad1_model.h5` at the end and replace the one in your project

In [ ]:
# ── Step 1: Upload your AtoZ_3.1.zip dataset ─────────────────────────────────
from google.colab import files
print('Select your AtoZ_3.1.zip file to upload...')
uploaded = files.upload()

In [ ]:
# ── Step 2: Unzip dataset ─────────────────────────────────────────────────────
import zipfile, os

zip_name = list(uploaded.keys())[0]
print(f'Unzipping {zip_name}...')
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('.')

# Find the dataset folder
DATA_DIR = None
for name in ['AtoZ_3.1', 'AtoZ_3.0']:
    if os.path.isdir(name):
        DATA_DIR = name
        break

if DATA_DIR is None:
    # Try finding it inside a subfolder
    for root, dirs, files_ in os.walk('.'):
        for d in dirs:
            if 'AtoZ' in d:
                DATA_DIR = os.path.join(root, d)
                break

print(f'Dataset found at: {DATA_DIR}')
letters = [l for l in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, l))]
print(f'Letters found: {sorted(letters)}')

In [ ]:
# ── Step 3: Install dependencies ─────────────────────────────────────────────
!pip install -q opencv-python-headless scikit-learn matplotlib

In [ ]:
# ── Step 4: Load & prepare dataset ───────────────────────────────────────────
import numpy as np
import cv2
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split

IMG_SIZE = 400
NUM_CLASSES = 8

LETTER_TO_GROUP = {
    'A':0,'E':0,'M':0,'N':0,'S':0,'T':0,
    'B':1,'D':1,'F':1,'I':1,'K':1,'R':1,'U':1,'V':1,'W':1,
    'C':2,'O':2,
    'G':3,'H':3,
    'L':4,
    'P':5,'Q':5,'Z':5,
    'X':6,
    'Y':7,'J':7,
}

images, labels = [], []
letter_counts = {}

all_letters = sorted(os.listdir(DATA_DIR))
for idx, letter in enumerate(all_letters):
    letter_dir = os.path.join(DATA_DIR, letter)
    if not os.path.isdir(letter_dir):
        continue
    letter_up = letter.upper()
    if letter_up not in LETTER_TO_GROUP:
        continue
    group = LETTER_TO_GROUP[letter_up]
    files_ = [f for f in os.listdir(letter_dir)
               if f.lower().endswith(('.jpg','.jpeg','.png'))]
    for fname in files_:
        img = cv2.imread(os.path.join(letter_dir, fname))
        if img is None:
            continue
        if img.shape[:2] != (IMG_SIZE, IMG_SIZE):
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        images.append(img)
        labels.append(group)
    letter_counts[letter_up] = len(files_)
    print(f'[{idx+1:02d}/{len(all_letters)}] {letter_up}: {len(files_)} images → group {group}')

X = np.array(images, dtype='float32') / 255.0
y = to_categorical(labels, NUM_CLASSES)
print(f'\nTotal: {len(X)} images  |  Shape: {X.shape}')

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)
print(f'Train: {len(X_train)}  |  Val: {len(X_val)}')

In [ ]:
# ── Step 5: Build improved model ─────────────────────────────────────────────
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Conv2D, MaxPooling2D, Flatten,
                                      Dense, Dropout, BatchNormalization, Input)
from tensorflow.keras.optimizers import Adam

model = Sequential([
    Input(shape=(IMG_SIZE, IMG_SIZE, 3)),

    Conv2D(32, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(2,2),

    Conv2D(128, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(2,2),

    Conv2D(128, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(2,2),

    Flatten(),
    Dense(512, activation='relu'),
    Dropout(0.5),
    Dense(256, activation='relu'),
    Dropout(0.4),
    Dense(NUM_CLASSES, activation='softmax'),
])

model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

In [ ]:
# ── Step 6: Train with augmentation ──────────────────────────────────────────
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import (ModelCheckpoint, EarlyStopping,
                                         ReduceLROnPlateau)

aug = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.10,
    height_shift_range=0.10,
    zoom_range=0.15,
    brightness_range=[0.6, 1.4],
    shear_range=5,
    horizontal_flip=False,
    fill_mode='nearest',
)

callbacks = [
    ModelCheckpoint(
        'cnn8grps_rad1_model.h5',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1,
    ),
    EarlyStopping(
        monitor='val_accuracy',
        patience=12,
        restore_best_weights=True,
        verbose=1,
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1,
    ),
]

BATCH_SIZE = 32
EPOCHS     = 60
steps      = len(X_train) // BATCH_SIZE

history = model.fit(
    aug.flow(X_train, y_train, batch_size=BATCH_SIZE),
    steps_per_epoch=steps,
    epochs=EPOCHS,
    validation_data=(X_val, y_val),
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
# ── Step 7: Results & plot ────────────────────────────────────────────────────
import matplotlib.pyplot as plt

best_acc = max(history.history['val_accuracy']) * 100
print(f'\n✅ Best validation accuracy: {best_acc:.2f}%')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.plot(history.history['accuracy'],     label='Train')
ax1.plot(history.history['val_accuracy'], label='Val')
ax1.set_title('Accuracy'); ax1.legend(); ax1.grid(True, alpha=0.3)
ax2.plot(history.history['loss'],     label='Train')
ax2.plot(history.history['val_loss'], label='Val')
ax2.set_title('Loss'); ax2.legend(); ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('training_results.png', dpi=120)
plt.show()
print('Plot saved.')

In [ ]:
# ── Step 8: Download the trained model ───────────────────────────────────────
from google.colab import files
print('Downloading model...')
files.download('cnn8grps_rad1_model.h5')
print('\nDone! Replace the cnn8grps_rad1_model.h5 in your project folder with this file.')